# extract_video_v2 — extração de frames corrigida (V1–V7 da spec 10)

Substitui `extract_video.ipynb` (v1, PRESERVADO como registro — decisão D3).
Diferenças em relação à v1, cada uma com o porquê:

- **V1** conversão BGR→RGB feita UMA vez, após a seleção dos frames (a v1 convertia
  dentro do loop e re-convertia o frame de fallback, devolvendo-o a BGR).
- **V2/D4** falha NÃO grava tensor de zeros (que, normalizado, era o pixel médio do
  ImageNet — dado plausível, não buraco). Falha → ID em `_missing.json`; o dataset
  falha alto no build (C4 do `video_core.py`).
- **V3** nenhum `except Exception` engolindo bugs de shape: os 64 frames têm o mesmo
  shape por construção (índices escolhidos DEPOIS de contar).
- **V4** leitura sequencial em 2 passes (`grab`/`retrieve`) — sem `cap.set` (salto
  impreciso) nem `CAP_PROP_FRAME_COUNT` (estimado, inválido sob VFR).
- **V5/D1** grava `uint8` [64,3,224,224] SEM rescale/normalize (9,6 MB/vídeo — os
  mesmos bytes da v1). A normalização ImageNet é aplicada no carregamento
  (`video_core.load_frames`). Resize/crop continuam DENTRO do processor → idênticos
  aos da v1 (V5-NORM). A célula 3 prova isso antes de qualquer extração.
- **V6** `SPLIT` é a única linha editável; caminhos derivados; nomes reais dos JSON
  mapeados explicitamente (treino `EXIST2026_training.json`, teste
  `EXIST2026_test_clean.json` — NÃO seguem o mesmo padrão). `_extraction_meta.json`
  gravado ao lado das features.
- **V7** grava em `frames/v2/{split}/` — o skip resume-safe volta a ser correto
  porque a árvore nasce vazia.

Rodar 2×: `SPLIT="training"` e `SPLIT="test"` (Restart & Run All em cada).

In [1]:
# P1-fix: os caminhos deste notebook sao RELATIVOS a EXIST_VIDEOS/ (plano 11 §3:
# "kernel do Jupyter em /home/a67606/Thesis/EXIST_VIDEOS ou cd ate la"). Se o kernel
# nascer noutro cwd, o assert do JSON da celula seguinte falha. Idempotente.
import os
from pathlib import Path

_TARGET = Path("/home/a67606/Thesis/EXIST_VIDEOS")
if Path.cwd() != _TARGET:
    assert _TARGET.is_dir(), f"diretorio do projeto nao encontrado: {_TARGET}"
    os.chdir(_TARGET)
assert Path("Dataset").is_dir(), f"cwd errado: {Path.cwd()} (sem Dataset/)"
print("cwd:", Path.cwd())


cwd: /home/a67606/Thesis/EXIST_VIDEOS


In [2]:
%pip install --quiet opencv-python-headless

Note: you may need to restart the kernel to use updated packages.


In [3]:
#!/usr/bin/env python
# coding: utf-8
"""
Extração de frames v2 para VideoMAE — ver célula 0 (markdown) para o mapa de
correções V1–V7. Salva [64, 3, 224, 224] uint8 (resize+crop do processor, SEM
rescale/normalize) em frames/v2/{split}/{id_EXIST}.pt.
"""
import json, platform, subprocess, sys, time, traceback
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import VideoMAEImageProcessorFast
import transformers

# ── Config ───────────────────────────────────────────────────────────────────
SPLIT = "test"        # ← A ÚNICA LINHA A EDITAR: "training" ou "test" (V6)

# V6: os JSON NÃO seguem o mesmo padrão de nome — mapeamento explícito, sem f-string.
DATASET_JSON = {
    "training": Path("./Dataset/training/EXIST2026_training.json"),
    "test":     Path("./Dataset/test/EXIST2026_test_clean.json"),
}
EXPECTED_N = {"training": 2524, "test": 674}   # contagens conhecidas (spec §"Estado")

VIDEO_DIR        = Path(f"./Dataset/{SPLIT}/videos/")
OUTPUT_DIR       = Path(f"./frames/v2/{SPLIT}/")           # V7: árvore versionada
JSON_PATH        = DATASET_JSON[SPLIT]
NUM_FRAMES_SAVED = 64        # D1: 64 uint8 = mesmos bytes de 16 float32 (9 633 792 B)
MODEL_NAME       = "MCG-NJU/videomae-base"                 # NÃO MUDAR (spec §4)
# ─────────────────────────────────────────────────────────────────────────────

assert SPLIT in DATASET_JSON, f"SPLIT inválido: {SPLIT!r}"
assert JSON_PATH.exists(), f"JSON não encontrado: {JSON_PATH}"
assert VIDEO_DIR.is_dir(), f"Diretório de vídeos não encontrado: {VIDEO_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

processor = VideoMAEImageProcessorFast.from_pretrained(MODEL_NAME)
# V5-NORM req.2: mean/std/size lidos DO PROCESSOR (nunca hardcodados) — vão para o meta
# e para o video_core.load_frames via _extraction_meta.json.
CROP_H = processor.crop_size["height"]
CROP_W = processor.crop_size["width"]
print(f"processor: size={processor.size} crop={processor.crop_size} "
      f"mean={processor.image_mean} std={processor.image_std}")

df = pd.read_json(JSON_PATH).transpose()   # mesmo parse da v1
assert len(df) == EXPECTED_N[SPLIT], \
    f"JSON de {SPLIT} tem {len(df)} itens; esperado {EXPECTED_N[SPLIT]} — PARAR e investigar"
print(f"{SPLIT}: {len(df)} itens → {OUTPUT_DIR}")

[transformers] `VideoMAEImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `VideoMAEImageProcessor` instead.


processor: size=SizeDict(height=None, width=None, longest_edge=None, shortest_edge=224, max_height=None, max_width=None) crop=SizeDict(height=224, width=224, longest_edge=None, shortest_edge=None, max_height=None, max_width=None) mean=(0.485, 0.456, 0.406) std=(0.229, 0.224, 0.225)
test: 674 itens → frames/v2/test


In [4]:
# ── Probe V5-NORM: prova, ANTES de extrair, que desligar rescale/normalize não
# altera o resize/crop, e mede o erro de armazenamento uint8.
# Decisão T2 do plano 11: mesma chamada de processor da v1, com
# do_rescale=False, do_normalize=False. Tolerância dura: 0,017 (spec §V5-NORM).
rng = np.random.default_rng(0)
NEEDS_CAST = False          # vira True se o processor devolver float no modo v2
worst = 0.0
for (h, w) in [(360, 640), (1920, 1080), (720, 720)]:   # paisagem, retrato TikTok, quadrado
    img = rng.integers(0, 256, (h, w, 3), dtype=np.uint8)
    full = processor(images=[img], return_tensors="pt")["pixel_values"]       # modo v1
    raw  = processor(images=[img], do_rescale=False, do_normalize=False,
                     return_tensors="pt")["pixel_values"]                     # modo v2
    if raw.dtype != torch.uint8:
        # Fallback: quantiza explicitamente. Erro máx = 0,5/255/0,225 ≈ 0,0087
        # no espaço normalizado — dentro da tolerância, mas REGISTRADO no meta.
        NEEDS_CAST = True
        raw = raw.round().clamp(0, 255).to(torch.uint8)
    mean = torch.tensor(processor.image_mean).view(1, 1, 3, 1, 1)
    std  = torch.tensor(processor.image_std).view(1, 1, 3, 1, 1)
    recon = (raw.float() / 255.0 - mean) / std     # exatamente o que load_frames fará
    diff = (recon - full).abs().max().item()
    worst = max(worst, diff)
    print(f"  {h}x{w}: shape v2={tuple(raw.shape)} dtype={raw.dtype} max|Δ|={diff:.2e}")

print(f"\nNEEDS_CAST={NEEDS_CAST}  pior max|Δ|={worst:.2e}")
assert worst <= 0.017, (
    "V5-NORM REPROVADO: a reconstrução (uint8/255−mean)/std diverge do modo v1 além "
    "da tolerância de quantização — o resize/crop MUDOU entre os modos. PARAR. "
    "NÃO extrair. Ver plano 11 §10 (modos de falha).")
if NEEDS_CAST:
    print("[AVISO] processor devolveu float no modo v2; usando cast explícito "
          "(erro ≤0,0087, dentro da tolerância). Registrado no meta como needs_cast=true.")

  360x640: shape v2=(1, 1, 3, 224, 224) dtype=torch.uint8 max|Δ|=2.38e-07
  1920x1080: shape v2=(1, 1, 3, 224, 224) dtype=torch.uint8 max|Δ|=2.38e-07
  720x720: shape v2=(1, 1, 3, 224, 224) dtype=torch.uint8 max|Δ|=2.38e-07

NEEDS_CAST=False  pior max|Δ|=2.38e-07


In [5]:
def count_decodable_frames(video_path: Path) -> int:
    """Passe 1 (V4): conta os frames REALMENTE decodificáveis, sequencialmente.

    `cap.grab()` demultiplexa+decodifica sem converter/copiar o frame — é a forma
    barata de contar. NUNCA usar CAP_PROP_FRAME_COUNT (estimado por duração×fps,
    inválido para VFR de TikTok) nem cap.set(POS_FRAMES) (salto aproximado
    para o keyframe mais próximo). Ver spec §V4.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        return 0
    n = 0
    while cap.grab():
        n += 1
    cap.release()
    return n


def retrieve_frames_at(video_path: Path, targets_sorted: list) -> dict:
    """Passe 2 (V4/T1): percorre sequencialmente com grab(); retrieve() SÓ nos
    índices alvo. Memória O(len(targets)) em resolução nativa (≤ ~400 MB para
    64 frames 1080×1920) — nunca o vídeo inteiro (V4-MEM).
    Devolve {índice: frame BGR nativo} com o que conseguiu decodificar.
    """
    got = {}
    cap = cv2.VideoCapture(str(video_path))
    pos, ti = 0, 0
    while ti < len(targets_sorted):
        if not cap.grab():
            break                      # fim real (ou truncamento) — sem inventar nada
        if pos == targets_sorted[ti]:
            ret, frame = cap.retrieve()
            if ret and frame is not None:
                got[pos] = frame
            ti += 1                    # alvo consumido mesmo se retrieve falhou
        pos += 1
    cap.release()
    return got


def sample_frames_v2(video_path: Path, n: int):
    """Amostragem uniforme sobre a contagem REAL (V4), sem fallbacks fabricados.

    Devolve (frames_rgb [n,H,W,3] uint8, info) ou (None, info) se o vídeo não
    tem nenhum frame decodificável (o chamador manda o ID para _missing.json — D4).

    - Índices escolhidos DEPOIS de contar → shapes iguais por construção; o
      np.stack não pode falhar por mistura com fallback 224×224 (V3).
    - Se um retrieve falhar (arquivos truncados: ~1–4 % do split de teste,
      medição do Agente 3 em 03_OPTIMIZATION_FINDINGS.md), preenche com o frame
      REAL mais próximo já decodificado — nunca zeros, e a contagem de
      preenchimentos é reportada (info["n_filled"]).
    - Conversão BGR→RGB UMA vez, no fim, só nos selecionados (V1).
    """
    total = count_decodable_frames(video_path)
    if total == 0:
        return None, {"reason": "0 frames decodificáveis no passe 1"}
    wanted = np.linspace(0, total - 1, n).round().astype(int)   # com repetição se total<n
    targets = sorted(set(wanted.tolist()))
    got = retrieve_frames_at(video_path, targets)
    if not got:
        return None, {"reason": f"passe 2 não decodificou nenhum frame (passe 1 contou {total})"}
    avail = sorted(got)
    n_filled = 0
    frames = []
    for idx in wanted:
        if idx in got:
            frames.append(got[idx])
        else:
            nearest = min(avail, key=lambda a: abs(a - idx))
            frames.append(got[nearest])
            n_filled += 1
    frames_bgr = np.stack(frames)                            # [n, H, W, 3] BGR — shapes iguais
    frames_rgb = np.ascontiguousarray(frames_bgr[..., ::-1]) # V1: UMA conversão, no fim
    return frames_rgb, {"total_frames": int(total), "n_filled": int(n_filled)}


def process_frames_v2(frames_rgb: np.ndarray) -> torch.Tensor:
    """V5/T2: MESMO processor da v1, com rescale/normalize desligados.
    Resize (aresta menor) + center-crop idênticos aos da v1; saída uint8.
    A normalização ImageNet acontece no carregamento (video_core.load_frames, C2).
    """
    processed = processor(
        images=[frames_rgb[i] for i in range(frames_rgb.shape[0])],
        do_rescale=False, do_normalize=False,
        return_tensors="pt",
    )
    pv = processed["pixel_values"].squeeze(0)                # [n, 3, H, W]
    if pv.dtype != torch.uint8:                              # fallback do probe (célula 3)
        pv = pv.round().clamp(0, 255).to(torch.uint8)
    assert pv.shape == (NUM_FRAMES_SAVED, 3, CROP_H, CROP_W), \
        f"shape inesperado: {tuple(pv.shape)}"
    return pv

In [6]:
# ── Projeção de custo ANTES da run completa (decisão T1 do plano 11).
# Critério: projeção > 3 h para os 3 198 vídeos → PARAR e reportar (reavaliar a
# opção (b) do V4-MEM ou paralelizar). Esta célula não grava nenhum arquivo.
probe_rows = df.head(10)
t0 = time.perf_counter()
for _, row in probe_rows.iterrows():
    vp = VIDEO_DIR / Path(str(row["path_video"])).name
    if not vp.exists():
        continue
    frames_rgb, info = sample_frames_v2(vp, NUM_FRAMES_SAVED)
    if frames_rgb is not None:
        _ = process_frames_v2(frames_rgb)
dt = time.perf_counter() - t0
per_item = dt / len(probe_rows)
proj_min = per_item * 3198 / 60
print(f"{per_item:.2f} s/vídeo → projeção p/ 3 198 vídeos: {proj_min:.0f} min "
      f"({proj_min/60:.1f} h)")
assert proj_min <= 180, (
    f"Projeção {proj_min:.0f} min > 180 min: PARAR e reportar (plano 11 §4.1). "
    "Não iniciar a run completa.")

0.65 s/vídeo → projeção p/ 3 198 vídeos: 35 min (0.6 h)


In [7]:
missing, warns = [], []
t_start = time.perf_counter()

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Frames v2 [{SPLIT}]"):
    item_id  = str(row["id_EXIST"])
    out_path = OUTPUT_DIR / f"{item_id}.pt"

    # Resume-safe — CORRETO na v2 porque a árvore nasce vazia (V7). Numa retomada
    # após queda, os itens já gravados são pulados e os que falharam são re-tentados
    # (falha não grava arquivo, então volta a entrar aqui).
    if out_path.exists():
        continue

    # SIMP-10: UMA resolução de caminho (o "fallback" da v1 era um no-op idêntico).
    video_path = VIDEO_DIR / Path(str(row["path_video"])).name
    if not video_path.exists():
        # V2/D4: NÃO gravar zeros (zeros normalizados = pixel médio do ImageNet =
        # dado plausível). Ausência vira manifesto; o dataset falha alto no build.
        missing.append({"id": item_id, "reason": f"vídeo não encontrado: {video_path.name}"})
        continue

    try:
        frames_rgb, info = sample_frames_v2(video_path, NUM_FRAMES_SAVED)
    except Exception as e:
        # Diferente do except da v1 (que devolvia None → zeros no disco, V3):
        # aqui a exceção é IMPRESSA INTEIRA e o item vai para o manifesto.
        traceback.print_exc()
        missing.append({"id": item_id, "reason": f"{type(e).__name__}: {e}"})
        continue

    if frames_rgb is None:
        missing.append({"id": item_id, "reason": info["reason"]})
        continue
    if info["n_filled"] > 0:
        warns.append({"id": item_id, **info})

    torch.save(process_frames_v2(frames_rgb), out_path)

elapsed_min = (time.perf_counter() - t_start) / 60
n_saved = len(list(OUTPUT_DIR.glob("*.pt")))
print(f"\n{SPLIT}: {n_saved} gravados | {len(missing)} faltantes | "
      f"{len(warns)} com preenchimento | {elapsed_min:.1f} min")

Frames v2 [test]:  42%|███████████████████████████▏                                    | 286/674 [04:30<05:21,  1.21it/s][h264 @ 0x30519000] Invalid NAL unit size (9911 > 7562).
[h264 @ 0x30519000] missing picture in access unit with size 7566
[h264 @ 0x305ac340] Invalid NAL unit size (9911 > 7562).
[h264 @ 0x305ac340] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x2700cdc0] stream 1, offset 0xbb4b0: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x2700cdc0] stream 0, offset 0xbb55f: partial file
[h264 @ 0x30434c00] Invalid NAL unit size (9911 > 7562).
[h264 @ 0x30434c00] missing picture in access unit with size 7566
[h264 @ 0x3043dd80] Invalid NAL unit size (9911 > 7562).
[h264 @ 0x3043dd80] Error splitting the input into NAL units.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x2700cdc0] stream 1, offset 0xbb4b0: partial file
Frames v2 [test]:  64%|█████████████████████████████████████████                       | 432/674 [06:22<02:58,  1.36it/s][h264 @ 0x3059d2c0] Invalid NAL unit siz


test: 674 gravados | 0 faltantes | 0 com preenchimento | 9.8 min


In [8]:
def _pip_version(pkg: str) -> str:
    try:
        out = subprocess.run([sys.executable, "-m", "pip", "show", pkg],
                             capture_output=True, text=True).stdout
        return next(l.split(":", 1)[1].strip() for l in out.splitlines()
                    if l.startswith("Version:"))
    except Exception:
        return "desconhecida"

meta = {
    # o que define o resultado (spec §V6, item 2) — NADA hardcodado:
    "split": SPLIT,
    "num_frames_saved": NUM_FRAMES_SAVED,
    "dtype": "uint8",
    "storage": "resize+center_crop do processor, SEM rescale e SEM normalize (V5); "
               "normalização ImageNet aplicada no carregamento (video_core.load_frames)",
    "needs_cast_fallback": bool(NEEDS_CAST),   # probe da célula 3
    "model_name": MODEL_NAME,
    "image_mean": list(processor.image_mean),  # lidos DO processor (req. V5-NORM 2)
    "image_std": list(processor.image_std),
    "size": dict(processor.size),
    "crop_size": dict(processor.crop_size),
    "processor_config": processor.to_dict(),
    "sampling": "2 passes sequenciais (grab/retrieve); índices = "
                "linspace(0, total_real-1, 64).round(); BGR→RGB uma vez após seleção",
    "n_items": int(len(df)),
    "n_saved": int(n_saved),
    "n_missing": int(len(missing)),
    "n_with_fill": int(len(warns)),
    "versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "opencv": cv2.__version__,
        "numpy": np.__version__,
        "opencv_pip": _pip_version("opencv-python-headless"),
    },
    "date": datetime.now(timezone.utc).isoformat(),
    "notebook": "extract_video_v2.ipynb",
}
with open(OUTPUT_DIR / "_extraction_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

# D4: manifesto SEMPRE gravado, mesmo vazio — ausência do arquivo passa a significar
# "extração não terminou", não "não houve falhas".
with open(OUTPUT_DIR / "_missing.json", "w") as f:
    json.dump({"split": SPLIT, "missing": missing, "date": meta["date"]}, f, indent=2)
with open(OUTPUT_DIR / "_extraction_warnings.json", "w") as f:
    json.dump({"split": SPLIT, "filled": warns, "date": meta["date"]}, f, indent=2)

print(f"meta/manifesto gravados em {OUTPUT_DIR}")
print(f"faltantes: {len(missing)}")
if missing:
    print("IDs:", [m["id"] for m in missing][:20])
    print("⚠ A v1 teve 0 falhas nestes dados (spec §V2, medido). QUALQUER faltante "
          "aqui é regressão do script — investigar antes de seguir (plano 11 §10).")
frac_fill = len(warns) / max(1, len(df))
if frac_fill > 0.05:
    print(f"⚠ {frac_fill:.1%} dos vídeos precisaram de preenchimento (>5%): "
          "PARAR e investigar (plano 11 §10).")

meta/manifesto gravados em frames/v2/test
faltantes: 0


In [9]:

import json
from pathlib import Path

# Teste 1 — metas de training e test idênticos exceto campos por-split
a = json.load(open("frames/v2/training/_extraction_meta.json"))
b = json.load(open("frames/v2/test/_extraction_meta.json"))
IGNORE = {"split", "n_items", "n_saved", "n_missing", "n_with_fill", "date"}
assert set(a) == set(b), set(a) ^ set(b)
diff = [k for k in set(a) - IGNORE if a[k] != b[k]]
print("teste 1 [frames]:", "PASS" if not diff else f"FAIL — divergem: {diff}")

# Teste 2 — contagem
EXPECT = {"training": 2524, "test": 674}
ok2 = True
for split, n_exp in EXPECT.items():
    n = len(list((Path("frames/v2") / split).glob("*.pt")))
    ok2 &= (n == n_exp)
    print(f"  frames/v2/{split}: {n} (esperado {n_exp})")
print("teste 2:", "PASS" if ok2 else "FAIL")

# Teste 3 — manifestos vazios + gate de 5% de preenchimento
ok3 = True
for split in ["training", "test"]:
    d = Path("frames/v2") / split
    miss = json.load(open(d / "_missing.json"))["missing"]
    fill = json.load(open(d / "_extraction_warnings.json"))["filled"]
    frac = len(fill) / EXPECT[split]
    ok3 &= (len(miss) == 0 and frac <= 0.05)
    print(f"  {split}: faltantes={len(miss)} preenchidos={len(fill)} ({frac:.1%})")
print("teste 3:", "PASS" if ok3 else "FAIL")

teste 1 [frames]: PASS
  frames/v2/training: 2524 (esperado 2524)
  frames/v2/test: 674 (esperado 674)
teste 2: PASS
  training: faltantes=0 preenchidos=0 (0.0%)
  test: faltantes=0 preenchidos=0 (0.0%)
teste 3: PASS
